## Exploring Experiment Results
This notebook allows for exploration of experiment results generated by scripts.

In [10]:
import pandas as pd
from sklearn.metrics import classification_report

In [11]:
# Experiment we want to explore - customise these
TARGET_LABEL = "object"
EXPERIMENT_NUMBER = 1
# Use 0 to display every row, or a 1-based row number within the experiment.
ROW_NUMBER = 3

In [12]:
# Generated constants
FOLDER_NAME = f"{TARGET_LABEL}_classify"
RESULTS_PATH = f"../results/{FOLDER_NAME}"
EXPERIMENTS_DF_PATH = f"{RESULTS_PATH}/experiments.parquet"

In [ ]:
def get_experiment_rows(df, experiment_number, row_number=0):
    """
    Return the requested 1-based row(s) within an experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Returns:
        pd.DataFrame: The selected row as a one-row DataFrame, or every row for the
            experiment when row_number is 0.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    # Filter the DataFrame for the specified experiment number
    df_experiment = df[df["experiment_number"] == experiment_number]

    # Catch errors for invalid experiment numbers or row numbers
    if df_experiment.empty:
        raise ValueError(f"Experiment number {experiment_number} not found in DataFrame.")
    if row_number < 0:
        raise ValueError("ROW_NUMBER must be 0 or a positive integer.")
    if row_number > len(df_experiment):
        raise ValueError(
            f"Row {row_number} not found in experiment {experiment_number}; "
            f"it contains {len(df_experiment)} row(s)."
        )
    # Return all rows if row_number is 0 or one row if a valid row_number is provided
    if row_number == 0:
        return df_experiment
    return df_experiment.iloc[[row_number - 1]]

In [ ]:
def print_experiment_info(df, experiment_number, row_number=0):
    """
    Print the model and data configurations, training history, and evaluation results for a
    given experiment number and optional 1-based row number. A row number of 0
    prints every row.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the information for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        # Print model and data configurations
        print(f"\n---- Row {display_row_number}: Model {row['model_type']} ----\n")
        print("Train Config:")
        for key, value in row["train_config"].items():
            print(f"  {key}: {value}")
        print("\nData Config:")
        for key, value in row["data_config"].items():
            print(f"  {key}: {value}")

        # Print training history
        history = row["history"]
        print("Training History:")
        for epoch_idx, metrics in enumerate(history):
            print(f"  Epoch {epoch_idx + 1}:")
            for metric_name, metric_value in metrics.items():
                print(f"    {metric_name}: {metric_value}")

        # Print evaluation results
        print("\nStandard Test Results:")
        print(f"  Test Accuracy: {row['test_acc']:.4f}")
        print(f"  Test Loss: {row['test_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_weighted_f1_avg']:.4f}")

        print("\nUnseen Matched Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_matched_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_matched_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_matched_weighted_f1_avg']:.4f}")

        print("\nUnseen Related Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_related_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_related_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_related_weighted_f1_avg']:.4f}")

In [ ]:
def print_classification_report(df, experiment_number, row_number=0):
    """
    Print classification reports for the standard test set. A row number of 0
    prints a report for every row in the selected experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Classification Reports for Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the classification report for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        print(f"\n---- Row {display_row_number}: Model {row['model_type']} ----\n")

        # Generate and print the classification report
        report = classification_report(
            row["test_y_true"], row["test_y_pred"], target_names=row["train_labels"]
        )
        print(report)

In [16]:
# Load the required experiment results from the Parquet file
df = pd.read_parquet(EXPERIMENTS_DF_PATH)
df

,experiment_number,experiment_name,seed,deterministic,freeze_backbone,pretrained_checkpoint,train_config,data_config,model_type,train_labels,...,test_unseen_matched_weighted_f1_avg,test_unseen_matched_y_true,test_unseen_matched_y_expected,test_unseen_matched_y_pred,test_unseen_related_acc,test_unseen_related_loss,test_unseen_related_weighted_f1_avg,test_unseen_related_y_true,test_unseen_related_y_expected,test_unseen_related_y_pred
0,1,finetuned_comparison,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",baseline,"[football, hammer, mug, plastic_bottle, scisso...",...,0.426408,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[3, 3, 3, 12, 3, 3, 3, 3, 3, 3, 3, 12, 3, 3, 3...",18.236111,8.432156,0.222964,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[8, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 0,..."
1,1,finetuned_comparison,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,"[football, hammer, mug, plastic_bottle, scisso...",...,0.482544,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",20.861111,5.715310,0.259053,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 1..."
2,1,finetuned_comparison,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",efficientnet_b0,"[football, hammer, mug, plastic_bottle, scisso...",...,0.494025,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 12, 3, 3, 3, 12...",20.888889,6.266372,0.252761,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 6, 11, 11, 11, 11..."
3,1,finetuned_comparison,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",vit_b_16,"[football, hammer, mug, plastic_bottle, scisso...",...,0.486557,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",22.319444,6.667398,0.256601,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 11, 0, 11, 11, 0,..."
4,1,finetuned_comparison,129,True,False,timm/deit_tiny_patch16_224.fb_in1k,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",deit_tiny,"[football, hammer, mug, plastic_bottle, scisso...",...,0.440350,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",20.138889,7.106839,0.241937,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 1..."
5,1,finetuned_comparison,129,True,False,alanz-mit/FoundationTactile@e1a16123575eb26e78...,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",t3_tiny,"[football, hammer, mug, plastic_bottle, scisso...",...,0.379150,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[3, 3, 3, 3, 12, 12, 12, 3, 3, 12, 12, 3, 3, 3...",11.388889,6.306660,0.149310,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 1..."


In [17]:
print_experiment_info(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Experiment Number: 1
Experiment Name: finetuned_comparison


---- Row 3: Model efficientnet_b0 ----

Train Config:
  checkpoint_dir: /content/drive/MyDrive/Colab Notebooks/touch-ex/checkpoints/object_classify/001
  learning_rate: 2e-05
  min_learning_rate: 1e-06
  model_title: efficientnet_b0
  momentum: 0.9
  num_epochs: 10
  optimizer: adamw
  warmup_epochs: 1
  warmup_start_factor: 0.1
  weight_decay: 0.02

Data Config:
  batch_size: 32
  bg_path: data/baseline.jpg
  filtered_force_level: None
  filtered_motion: None
  norm_cache_path: configs/norm_cache.json
  norm_type: dataset
  num_workers: 8
  random_state: 129
  shuffle_map: {'test': False, 'train': True, 'val': False}
  split_size: 0.2
  stratify_label: object
  transform_name: pad_224
Training History:
  Epoch 1:
    epoch: 1
    learning_rate: 2e-05
    train_acc: 52.39557485525228
    train_loss: 1.601840622346014
    val_acc: 83.87788778877888
    val_loss: 0.47858296571628717
  Epoch 2:
    epoch: 2
    learning_rate: 1.

In [18]:
print_classification_report(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Classification Reports for Experiment Number: 1
Experiment Name: finetuned_comparison


---- Row 3: Model efficientnet_b0 ----

                precision    recall  f1-score   support

      football       0.97      0.97      0.97       960
        hammer       0.90      0.85      0.88       840
           mug       0.86      0.85      0.86       840
plastic_bottle       0.98      0.99      0.99       840
      scissors       0.85      0.88      0.87       840
sponge_scourer       1.00      0.99      1.00       840
     tea_towel       0.95      1.00      0.97       840
   tennis_ball       0.99      1.00      1.00       960
     tin_beans       0.93      0.71      0.80       840
   toilet_roll       0.99      0.89      0.94       840
    toothbrush       1.00      0.89      0.94       840
 tube_pringles       0.69      0.93      0.79       840
     tv_remote       0.97      0.99      0.98       960
  wooden_spoon       0.94      0.97      0.96       840

      accuracy                